In [12]:
import pandas as pd
import numpy as np
import re


np.random.seed(42)

# Define organisms and antibiotics
organisms = [
    "escherichia coli",
    "klebsiella pneumoniae",
    "proteus mirabilis",
    "pseudomonas aeruginosa"
]

antibiotics = [
    "amikacin", "amoxicillin", "augmentin", "cefepime", "cefotaxime",
    "cefoxitin", "cefpodoxime", "ceftazidime", "ertapenem",
    "gentamicin", "imipenem", "meropenem", "nitrofurantoin",
    "piperacillin_tazobactam", "tazocin", "temocillin", "tigecycline"
]

# Possible outcomes
results = ["susceptible", "resistant", "intermediate"]

# Function to randomly assign result or NaN (not tested)
def random_result(p_nan=0.4):  # 40% chance not tested
    if np.random.rand() < p_nan:
        return np.nan
    return np.random.choice(results, p=[0.7, 0.25, 0.05])

# Number of samples
n_samples = 200

# Create base dataframe
antibiogram_df = pd.DataFrame({
    "lab_test_id": [f"id_{i}" for i in range(n_samples)],
    "organism": np.random.choice(organisms, size=n_samples)
})

# Add antibiotic columns
for ab in antibiotics:
    antibiogram_df[ab] = [random_result() for _ in range(n_samples)]

antibiogram_df.head()

,lab_test_id,organism,amikacin,amoxicillin,augmentin,cefepime,cefotaxime,cefoxitin,cefpodoxime,ceftazidime,ertapenem,gentamicin,imipenem,meropenem,nitrofurantoin,piperacillin_tazobactam,tazocin,temocillin,tigecycline
0,id_0,proteus mirabilis,NaN,resistant,susceptible,susceptible,susceptible,NaN,susceptible,susceptible,susceptible,resistant,NaN,susceptible,susceptible,resistant,resistant,susceptible,NaN
1,id_1,pseudomonas aeruginosa,susceptible,NaN,NaN,resistant,susceptible,susceptible,susceptible,NaN,susceptible,NaN,resistant,intermediate,susceptible,NaN,resistant,susceptible,susceptible
2,id_2,escherichia coli,resistant,susceptible,NaN,susceptible,NaN,susceptible,NaN,resistant,susceptible,NaN,NaN,susceptible,susceptible,NaN,NaN,resistant,susceptible
3,id_3,proteus mirabilis,NaN,susceptible,susceptible,susceptible,susceptible,NaN,susceptible,NaN,NaN,susceptible,NaN,NaN,susceptible,intermediate,susceptible,NaN,susceptible
4,id_4,proteus mirabilis,resistant,susceptible,susceptible,susceptible,susceptible,susceptible,intermediate,NaN,NaN,intermediate,susceptible,susceptible,NaN,NaN,NaN,NaN,NaN


In [ ]:
# standardise column values into S, R and I 

def standardise_result(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).lower().strip()
    
    if "resistant" in x:
        return "R"
    
    if "intermediate" in x:
        return "I"
    
    if "susceptible" in x or "sensitive" in x:
        return "S"
    
    if "optimised dosing" in x:
        return "I"  

In [ ]:
mic_breakpoints = {
    "amikacin": (8, 8),
    "amoxicillin": (8, 8),
    "augmentin": (8, 8),  # amoxicillin-clavulanate (simplified)
    "cefepime": (1, 4),
    "cefotaxime": (1, 2),
    "cefoxitin": (8, 8),
    "cefpodoxime": (1, 1),  # UTI/oral only
    "ceftazidime": (1, 4),
    "ceftriaxone": (1, 2),
    "cefuroxime": (8, 8),  # oral/IV varies; simplified
    "cephalexin": (8, 8),  # oral only
    "ciprofloxacin": (0.25, 0.5),
    "co-amoxiclav": (8, 8),  # same as augmentin
    "co-trimoxazole": (2, 4),
    "ertapenem": (0.5, 0.5),
    "esbl_markers (ss = present)": None,  # not an MIC → keep as None
    "gentamicin": (2, 2),
    "imipenem": (2, 4),
    "meropenem": (2, 8),
    "nitrofurantoin": (64, 64),  # E. coli UTI only
    "piperacillin-tazobactam": (8, 8),
    "tazocin": (8, 8),  # synonym
    "temocillin": (0.001, 16),  # UTI only
    "tigecycline": (0.5, 0.5),
}


In [ ]:
def parse_mic(x):
    if pd.isna(x):
        return None, None
    
    x = str(x).strip().lower()
    
    match = re.match(r"^(<=|>=|<|>)?\s*([0-9]*\.?[0-9]+)$", x) #captures optional operator and numeric value in 2 groups
    
    if not match:
        return None, None
    
    operator = match.group(1) or "="
    value = float(match.group(2))
    
    return operator, value


def classify_mic_result(x, s_breakpoint, r_breakpoint):
    if pd.isna(x):
        return np.nan
    
    # Keep already-standardised categorical results
    x_str = str(x).strip().lower()
    
    if "resistant" in x:
        return "R"
    
    if "intermediate" in x:
        return "I"
    
    if "susceptible" in x or "sensitive" in x:
        return "S"
    
    if "optimised dosing" in x:
        return "I"  
    
    op, mic_value = parse_mic(x)
    
    if mic_value is None:
        return np.nan
    
    # Exact or upper-bound MICs
    if op in ["=", "<=", "<"]:
        if mic_value <= s_breakpoint:
            return "S"
        elif mic_value > r_breakpoint:
            return "R"
        else:
            return "I"
    
    # Lower-bound MICs
    if op in [">", ">="]:
        if mic_value > r_breakpoint:
            return "R"
        elif mic_value <= s_breakpoint:
            # This is ambiguous: e.g. >0.25 when S <=0.25
            return np.nan
        else:
            return "I"
    
    return np.nan


def map_mics_to_susceptibility(df, mic_breakpoints):
    df = df.copy()
    
    for ab, breakpoints in mic_breakpoints.items():
        if breakpoints is None:
            continue
            
        s_breakpoint, r_breakpoint = breakpoints
        
        if ab in df.columns:
            df[ab] = df[ab].apply(
                lambda x: classify_mic_result(x, s_breakpoint, r_breakpoint)
            )
    
    return df